# Introducción a Function Calling

### A. Introducción

In [56]:
def capital_Luna():

    info = "La Luna tiene su capital que es Lunar City !"

    return info

In [57]:
i = capital_Luna()
print(i)

La Luna tiene su capital que es Lunar City !


### Cargar el modelo

In [58]:
from langchain.chat_models import init_chat_model

from dotenv import load_dotenv
import os

# Generar una funcion que obtenga el modelo LLM

def get_llm(prefix_model, temperature=0.9):
    """
    Obtiene un modelo de lenguaje (LLM) basado en el modelo especificado.

    Args:
        model (str): El nombre del modelo a utilizar.
        temperature (float, optional): La temperatura para la generación de texto. 
                                        Valores más altos producen respuestas más creativas.
                                        Valores más bajos producen respuestas más deterministas.
                                        Por defecto es 0.9.

    Returns:
        llm: Un objeto de modelo de lenguaje inicializado.
    """
    llm = init_chat_model(
                            model=prefix_model, 
                            # temperature=temperature
                          )
    return llm

# Configuración del modelo de lenguaje
load_dotenv()

prefix_model = os.getenv("PREFIX_MODEL")

# Obtención del modelo
llm = get_llm(prefix_model)



### Usar el modelo

In [59]:
# 5. Usar el LLM

PREGUNTA = "Cual es la capital de la Luna?"

response = llm.invoke(PREGUNTA)

print(response.content)

[{'type': 'text', 'text': 'La Luna **no tiene capital**, ya que no está habitada por seres humanos ni tiene gobiernos, países o ciudades. \n\nSin embargo, desde el punto de vista científico y de la exploración espacial, hay lugares que han tenido mucha importancia:\n\n* **Base Tranquilidad (Tranquility Base):** Es el lugar donde el Apolo 11 alunizó por primera vez en 1969. Si la Luna tuviera una "capital histórica", sin duda sería esta.\n* **El Polo Sur lunar:** Actualmente es el "lugar de más interés" porque allí se busca agua helada, y varias agencias espaciales planean construir allí las primeras bases científicas permanentes.', 'extras': {'signature': 'EmAKXgFpFH0TAN+S8M2U4qqDnivj4xjwKq1v1BxyhskbZ/6vbABcbltYqUk+f8xLS13lYlbnhxDbMb+zVvDnZu7fm9ypy2L9RNSqH64i4jWYptNp7FKJgC3dMHOZuHuPiC4='}}]


### B. Pasos para crear una Function Calling

- Paso 1 : Definir una función
- Paso 2 : Definir la función como una herramienta
- Paso 3 : Vincularlo con el modelo

#### Paso 1

In [60]:
def capital_Luna():

    info = "La Luna tiene su capital que es Lunar City !"

    return info

#
print(capital_Luna)


<function capital_Luna at 0x128bd6160>


#### Paso 2

In [61]:
from langchain.tools import tool

@tool
def capital_Luna():
    """
      Consulta informacion de la Luna, como puede ser la capital de la Luna
    """
    info = "La Luna tiene su capital que es Lunar City !"

    return info

#
print("Nombre:", capital_Luna.name)
print("Descripción:", capital_Luna.description)
print("Argumentos:", capital_Luna.args)

Nombre: capital_Luna
Descripción: Consulta informacion de la Luna, como puede ser la capital de la Luna
Argumentos: {}


#### Paso 3

In [62]:
from langchain_core.messages import HumanMessage
from langchain.agents import create_agent


agente_basico = create_agent(
    model = llm,
    tools = [
                capital_Luna
            ]
)

# Hacer la consulta

consulta = {"messages": [HumanMessage(content="Cual es la capital de la Luna?")]}

respuesta = agente_basico.invoke(consulta)

print(respuesta)



{'messages': [HumanMessage(content='Cual es la capital de la Luna?', additional_kwargs={}, response_metadata={}, id='6a1fd5fa-e732-4421-8504-5aec81782fe7'), AIMessage(content=[], additional_kwargs={'function_call': {'name': 'capital_Luna', 'arguments': '{}'}, '__gemini_function_call_thought_signatures__': {'call_2390361': 'EmAKXgFpFH0T9JEw3mf8cphEvWSkp3C9Aem+GnyBaFkfdxu7G16ylyIHc9quMT+ZDvUbMRmZZXFRilBB9wYiWoglZd10G3gRJPiVgXyv1txSGcH8YRWnm0Z/SXRnVVuEqsU='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a14ae1-4977-7280-b5a6-2ad489263f64-0', tool_calls=[{'name': 'capital_Luna', 'args': {}, 'id': 'call_2390361', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 37, 'output_tokens': 10, 'total_tokens': 47, 'input_token_details': {'cache_read': 0}}), ToolMessage(content='La Luna tiene su capital que es Lunar City !', name='capital_Luna', id='740d712b-7521-

In [63]:
    for m in respuesta["messages"]:
        print(f"[{type(m).__name__}] {m.text or m.tool_calls}")

[HumanMessage] Cual es la capital de la Luna?
[AIMessage] [{'name': 'capital_Luna', 'args': {}, 'id': 'call_2390361', 'type': 'tool_call'}]
[ToolMessage] La Luna tiene su capital que es Lunar City !
[AIMessage] La capital de la Luna es **Lunar City**.


### Ejercicio 1

Crear un herramienta nueva que me permita obtener información de la capital del planeta Marte

In [64]:
from langchain.tools import tool

@tool
def capital_Marte():
    """
      Consulta informacion del planeta Marte, como puede ser la capital de Marte
    """
    info = "El planeta marte tiene su capital que es Olimpus City !"

    return info


agente_basico = create_agent(
    model = llm,
    tools = [
                capital_Luna,
                capital_Marte
            ]
)

# Hacer la consulta

consulta = {"messages": [HumanMessage(content="Cual es la capital del planeta Marte?")]}

respuesta = agente_basico.invoke(consulta)

for m in respuesta["messages"]:
    print(f"[{type(m).__name__}] {m.text or m.tool_calls}")

[HumanMessage] Cual es la capital del planeta Marte?
[AIMessage] [{'name': 'capital_Marte', 'args': {}, 'id': 'call_2635279', 'type': 'tool_call'}]
[ToolMessage] El planeta marte tiene su capital que es Olimpus City !
[AIMessage] La capital del planeta Marte es **Olimpus City**.


### C. Pasos para crear una Function Calling con parámetros


In [65]:
@tool
def info_Marte(consulta: str ) -> str:
    """
      Consulta la informacion del planeta Marte, usalo cuando te pregunten sobre
      - capital
      - población 
      - pbi
    """
    info = {
        "capital" :"La capital es Olimpus City !",
        "poblacion" : "1 habitante",
        "pbi" : " 1 sol" 
    }

    consulta = consulta.lower()
    print(f"consulta {consulta}")
    print(f"dato 1 {info.get(consulta)}")

    return info.get(consulta)



print("Nombre:", info_Marte.name)
print("Descripción:", info_Marte.description)
print("Argumentos:", info_Marte.args)

Nombre: info_Marte
Descripción: Consulta la informacion del planeta Marte, usalo cuando te pregunten sobre
- capital
- población 
- pbi
Argumentos: {'consulta': {'title': 'Consulta', 'type': 'string'}}


In [66]:
agente_basico = create_agent(
    model = llm,
    tools = [
                capital_Luna,
                info_Marte
            ]
)

In [67]:
# Hacer la consulta

consulta = {"messages": [HumanMessage(content="Cual es la población del planeta Marte?")]}

respuesta = agente_basico.invoke(consulta)

for m in respuesta["messages"]:
    print(f"[{type(m).__name__}] {m.text or m.tool_calls}")

consulta poblacion
dato 1 1 habitante
[HumanMessage] Cual es la población del planeta Marte?
[AIMessage] [{'name': 'info_Marte', 'args': {'consulta': 'poblacion'}, 'id': 'call_4257957', 'type': 'tool_call'}]
[ToolMessage] 1 habitante
[AIMessage] La población del planeta Marte es de 1 habitante.


### Ejercicio 2

Modificar la herramienta de capital_Luna a info_Luna y que me permita obtener informacion de la Luna de su área, población y presidente